In [1]:
import pandas as pd
import numpy as np
import torch
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.linear_model import LinearRegression


## Loading Dataset

In [2]:
df = pd.read_csv('../dataset/insurance.csv')

In [3]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## Checking Data Quality

### Missing Values

In [4]:
missing_df = df.isnull().sum().reset_index()
missing_df.columns = ['Column', 'Missing Count']    # changing name of columns

# creating column 'Missing Percentage'
missing_df['Missing Percentage'] = (
    missing_df['Missing Count'] / len(df) * 100
)

fig = px.bar(
    missing_df,
    x = 'Column',
    y = 'Missing Count',
    hover_data = 'Missing Percentage'
)

fig.update_traces(
    hovertemplate = 
    "<b> %{x} </b> <br>" + 
    "Missing Count = %{y} <br>" +
    "Missing Percentage = %{customdata[0]:.2f}%"
)
fig.show()

### Duplicate Rows

In [5]:
df.duplicated().sum()

np.int64(1)

In [6]:
df.drop_duplicates(inplace=True)

In [7]:
df.duplicated().sum()

np.int64(0)

## Encoding

In [8]:
ordinal_encoder = OrdinalEncoder(categories=[['female', 'male'], ['yes', 'no']])
df[['sex', 'smoker']] = ordinal_encoder.fit_transform(df[['sex', 'smoker']])

In [9]:
df = pd.get_dummies(df, columns=['region'], dtype=int)

In [10]:
df.head()

,age,sex,bmi,children,smoker,charges,region_northeast,region_northwest,region_southeast,region_southwest
0,19,0.0,27.900,0,0.0,16884.92400,0,0,0,1
1,18,1.0,33.770,1,1.0,1725.55230,0,0,1,0
2,28,1.0,33.000,3,1.0,4449.46200,0,0,1,0
3,33,1.0,22.705,0,1.0,21984.47061,0,1,0,0
4,32,1.0,28.880,0,1.0,3866.85520,0,1,0,0


## Train Test Split

In [11]:
X = df.drop(columns=['charges'])
y = df[['charges']]

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    random_state = 42
)

## Feature Scaling

In [13]:
# Creating scalers
x_scaler = StandardScaler()
y_scaler = StandardScaler()

# Fitting training data
X_train = x_scaler.fit_transform(X_train)
X_test = x_scaler.transform(X_test)

y_train = y_scaler.fit_transform(y_train)
y_test = y_scaler.transform(y_test)   

In [14]:
y_train = y_train.ravel()
y_test = y_test.ravel()

## Linear Regression

In [15]:
class LinearRegression:

    def __init__(self, lr = 0.01, iterations=1000):
        self.lr = lr
        self.iterations = iterations
        self.loss_history = []

        # for visualizations
        self.training_history = {}
        self.weight_history = []
    
    def mean_squared_error(self, y_hat, y):
        m = len(y)
        return 1/m * (np.sum((y_hat-y)**2))
    
    def fit(self, X, y):

        m, n = X.shape
        self.weights = np.zeros(n)
        self.bias = 0


        for i in range(self.iterations):
            y_hat = np.dot(X, self.weights) + self.bias

            # loss history
            self.loss_history.append(self.mean_squared_error(y_hat, y))

            # saving prediction line
            if i % 50==0:
                self.training_history[i] = y_hat.copy()

            # saving weights
            self.weight_history.append(self.weights.copy())

            # slopes
            dloss_dweights = 1/m * (np.dot(X.T, y_hat-y))
            dloss_dbias = 1/m * (np.sum(y_hat-y))

            # parameters update
            self.weights -= self.lr * dloss_dweights
            self.bias -= self.lr * dloss_dbias

    def predict(self, X):
        predictions = np.dot(X, self.weights) + self.bias

        return predictions

    

In [16]:
model = LinearRegression()

In [17]:
model.fit(X_train, y_train)

In [18]:
preds = model.predict(X_test)

## Visualizing Training

In [19]:
frames = []

# y_train should be 1D
y_true = y_train.ravel()

for iteration, preds in model.training_history.items():

    temp = pd.DataFrame({
        "Actual": y_true,
        "Predicted": preds,
        "Iteration": iteration
    })

    frames.append(temp)

history_df = pd.concat(frames, ignore_index=True)

In [20]:
fig = px.scatter(
    history_df,
    x="Actual",
    y="Predicted",
    animation_frame="Iteration",
    title="Actual vs Predicted During Training",
    opacity=0.7
)

# Perfect prediction line
min_val = min(history_df["Actual"].min(), history_df["Predicted"].min())
max_val = max(history_df["Actual"].max(), history_df["Predicted"].max())

fig.add_shape(
    type="line",
    x0=min_val,
    y0=min_val,
    x1=max_val,
    y1=max_val,
    line=dict(color="red", dash="dash")
)

fig.update_layout(
    xaxis_title="Actual",
    yaxis_title="Predicted"
)

fig.write_image("../visualizations/training_animation.png")

In [21]:
feature_names = X.columns
weights_df = pd.DataFrame(
    model.weight_history,
    columns=feature_names
)

weights_df["Iteration"] = weights_df.index

weights_df = weights_df.melt(
    id_vars="Iteration",
    var_name="Feature",
    value_name="Weight"
)

fig = px.line(
    weights_df,
    x="Iteration",
    y="Weight",
    color="Feature",
    title="Feature Weights During Training"
)

fig.write_image("../visualizations/weights_plot.png")

## Loss Curve

In [22]:
fig = px.line(
    model.loss_history
)

fig.show()

## Checking Model Performance

### mean squared error

In [30]:
def get_mse(y_true, y_hat):
    m = len(y_true)
    return ((y_hat-y_true)**2).mean()

In [31]:
mse = get_mse(y_test, preds)

In [33]:
print("Scratch MSE : ", mse)

Scratch MSE :  0.2564786221379031


### r2

In [34]:
def get_r2(y_true, y_hat):
    SSR = np.sum((y_true - y_hat)**2)
    SST = np.sum((y_true - y_true.mean())**2)

    return 1 - (SSR/SST)

In [35]:
r2 = get_r2(y_test, preds)

In [36]:
print("Scratch r2: ", r2)

Scratch r2:  0.7959205069783579


## Sklearn Implementation

In [45]:
sklearn_model = LinearRegression()

In [46]:
sklearn_model.fit(X_train, y_train)

In [47]:
sklearn_preds = sklearn_model.predict(X_test)

In [48]:
sklearn_mse = get_mse(y_test, sklearn_preds)

In [49]:
print("Sklearn MSE: ", sklearn_mse)

Sklearn MSE:  0.2564786221379031


In [50]:
sklearn_r2 = get_r2(y_test, sklearn_preds)

In [51]:
print("Scratch r2: ", sklearn_r2)

Scratch r2:  0.7959205069783579


## Implementation Using PyTorch

In [60]:
class PyTorchLinearRegression:

    def __init__(self, lr=0.01, iterations=1000):
        self.lr = lr
        self.iterations = iterations
        self.loss_history = []

    def mean_squared_error(self, y_hat, y):
        return ((y_hat-y)**2).mean()

    def fit(self, X, y):
        m, n = X.shape
        self.weights = torch.zeros(n, requires_grad=True)
        self.bias = torch.zeros(1, requires_grad=True)
        X = torch.tensor(X, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)

        for epoch in range(self.iterations):
            
            y_hat = X @ self.weights + self.bias
            loss = self.mean_squared_error(y_hat, y)

            # backpropagation
            loss.backward()

            # weights update
            with torch.no_grad():
                self.weights -= self.lr * self.weights.grad
                self.bias -= self.lr * self.bias.grad

            # clear weights
            self.weights.grad.zero_()
            self.bias.grad.zero_()

    def predict(self, X):
        X = torch.tensor(X, dtype=torch.float32)
        y_hat = X @ self.weights + self.bias

        return y_hat.detach().numpy()

In [61]:
torch_model = PyTorchLinearRegression()

In [62]:
torch_model.fit(X_train, y_train)

In [64]:
torch_preds = torch_model.predict(X_test)

In [65]:
torch_mse = get_mse(y_test, torch_preds)

In [66]:
print("PyTorch MSE: ", torch_mse)

PyTorch MSE:  0.2564539747251465


In [67]:
torch_r2 = get_mse(y_test, torch_preds)

In [68]:
print("PyTorch r2: ", torch_r2)

PyTorch r2:  0.2564539747251465
